In [2]:
"""Clinical (% ED visits) vs. wastewater signal (4-week rolling mean) time
series, restricted to  samples that were tested
by both qPCR and sequencing 

Produces two chart families per virus (SARS-CoV-2, Flu A+B combined, RSV A+B
combined): one using qPCR copies/mL, one using sequencing rpkmf -- both
computed from the exact same matched sample cohort.

Missing weeks are bridged with a dashed segment rather than a solid line or
a blank gap.
"""

import math
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from matplotlib.ticker import FixedLocator, FuncFormatter
from scipy.stats import pearsonr, spearmanr

#Path to data
DATA_PATH = "Supp_Data_2_Clinical_Concordant_Data_final.xlsx"

COLOR_WW = "#3D2412"   # (brown) -- wastewater signal
COLOR_ED = "#2a78d6"   #  (blue) -- clinical (% ED visits)

GAP_LINESTYLE = (0, (4, 3))  # dashed, used by plot_gap_aware_series for missing-period bridges
FLOOR_LINESTYLE = (0, (1, 2))  # dots, used for segments sitting flat at the series' zero-substituted floor

VIRUSES = [
    {"sheet": "SARS-COV-2", "perc_col": "PERC_COVID", "title": "SARS-CoV-2", "slug": "sars_cov2"},
    {"sheet": "FLU", "perc_col": "PERC_INFLUENZA", "title": "Influenza (A+B combined)", "slug": "flu", "box_side": "left"},
    {"sheet": "RSV", "perc_col": "PERC_RSV", "title": "RSV (A+B combined)", "slug": "rsv", "box_side": "left"},
]

METRICS = [
    {
        "key": "copiesperml",
        "value_col": lambda v: "covid_copiesperml_rolling_4wk" if v["slug"] == "sars_cov2" else f"{v['slug']}_combined_copiesperml_rolling_4wk",
        "ylabel": "Wastewater qPCR copies/mL \n(4-week rolling mean)",
        "series_label": "Wastewater qPCR copies/ml \n(4-week rolling mean)",
        "out_suffix": "qpcr_concordant_vs_clinical_rolling_4wk_nodallas",
    },
    {
        "key": "rpkmf",
        "value_col": lambda v: "rpkmf_rolling_4wk" if v["slug"] == "sars_cov2" else f"{v['slug']}_combined_rpkmf_rolling_4wk",
        "ylabel": "Sequencing RPKMF \n(4-week rolling mean)",
        "series_label": "Sequencing RPKMF \n(4-week rolling mean)",
        "out_suffix": "sequencing_concordant_vs_clinical_rolling_4wk_nodallas",
    },
]

#Fucntion to calculate correaltion between weekly detections.
def compute_correlations(sub, perc_col, value_col):
    
    paired = sub[[perc_col, value_col]].dropna()
    n = len(paired)
    if n < 2:
        return None, None, None, None, n
    
    pearson_r, pearson_p = pearsonr(paired[perc_col], paired[value_col])
    spearman_r, spearman_p = spearmanr(paired[perc_col], paired[value_col])
    
    return pearson_r, pearson_p, spearman_r, spearman_p, n

#Plotting function of overlaid time-series charts
def plot_gap_aware_series(ax, x, y, color, label=None, linewidth=3.25, marker=None, markersize=None, zorder=3, floor_mask=None):
    """Plot a period-indexed series (x = one row per period on a regular
    weekly grid, already sorted; y may be NaN for periods with no data) as
    solid segments between temporally-adjacent known points, and a dashed
    segment (GAP_LINESTYLE) bridging directly across any run of one or more
    missing periods, so a real gap reads as "unknown" rather than a plain
    connected line or a blank stretch.

    floor_mask, if given, is a boolean sequence aligned with y marking points
    whose real value was an actual zero (displayed at the series' nonzero
    floor instead of true 0). A segment where BOTH endpoints are floor points
    is drawn with FLOOR_LINESTYLE (dotted) instead of solid, since it
    represents a "nothing detected" flat run rather than an observed trend
    sitting at that level. A segment touching a floor point on only one end
    stays solid.
    """
    
    x = list(x)
    y = list(y)
    floor_mask = list(floor_mask) if floor_mask is not None else [False] * len(y)
    known = [i for i, v in enumerate(y) if pd.notna(v)]

    if label is not None:
        ax.plot([], [], color=color, linewidth=linewidth, marker=marker,
                 markersize=markersize, linestyle="-", label=label, zorder=zorder)

    # Group consecutive same-style segments into ONE polyline per run rather
    # than one plot() call per week-to-week pair -- a dash/dot pattern only
    # cycles correctly across a continuous artist. A multi-week flat floor
    # run drawn as many separate 1-week-long segments each restarts the dash
    # phase from 0, and with round end-caps on a segment barely longer than
    # one dash period, each mini-segment renders almost fully "on" -- the
    # whole run then looks solid instead of dotted.
    runs = []  # (style, [point indices])
    for a, b in zip(known[:-1], known[1:]):
        if b - a != 1:
            style = GAP_LINESTYLE
        elif floor_mask[a] and floor_mask[b]:
            style = FLOOR_LINESTYLE
        else:
            style = "-"
        if runs and runs[-1][0] == style and runs[-1][1][-1] == a:
            runs[-1][1].append(b)
        else:
            runs.append((style, [a, b]))

    for style, idxs in runs:
        ax.plot([x[i] for i in idxs], [y[i] for i in idxs], color=color, linewidth=linewidth,
                 marker=marker, markersize=markersize, linestyle=style, zorder=zorder,
                 dash_capstyle="round" if style == FLOOR_LINESTYLE else "butt")


def _style_axes(ax_left, ax_right, left_ylabel):
    ax_left.set_ylabel(left_ylabel, fontsize=14, color="black", fontweight="bold")
    ax_left.tick_params(axis="y", colors="black")
    ax_left.spines["left"].set_color("black")

    ax_right.set_ylabel("% ED visits (Statewide)", fontsize=18, color="black", fontweight="bold")
    ax_right.tick_params(axis="y", colors="black")
    ax_right.spines["right"].set_color("black")
    ax_right.spines["left"].set_visible(False)

    ax_left.spines["top"].set_visible(False)
    ax_right.spines["top"].set_visible(False)
    ax_left.spines["right"].set_visible(False)

    ax_left.set_ylim(bottom=0)
    ax_right.set_ylim(bottom=0)

    ax_left.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax_left.set_axisbelow(True)


def _format_p(p):
    return f"p = {p:.2e}"


def _configure_symlog_axis(ax, values, top, mark_floor=True):
    """Symlog y-scale sized to `values`' own floor (nonzero min) and `top`.
    When `mark_floor` is True, 0 and the floor each get an explicit labeled
    tick; when False, the floor only sizes the symlog scale itself (so the
    low end doesn't collapse into the near-zero linear band) without
    surfacing it as its own tick/label -- used for the % ED axis, which has
    no real "limit of detection" concept, unlike qPCR/sequencing signal.

    Applied to BOTH axes (wastewater signal AND % ED visits). Pairing one log-scaled
    axis with a linear one left the linear series pinned near the bottom at
    its own troughs while the log series stayed elevated, so well-correlated
    series still occupied different vertical bands. Matching both axes to
    their own symlog scale lets proportionally similar swings render at
    proportionally similar heights on both, which is what actually makes
    them visually overlap.
    """
    nonzero = values[values > 0]
    floor = nonzero.min() if len(nonzero) else 1
    linthresh = floor if floor > 0 else 1
    ax.set_yscale("symlog", linthresh=linthresh)
    ax.set_ylim(0, top)

    default_ticks = [t for t in ax.get_yticks() if floor < t <= top]
    all_ticks = sorted(set(default_ticks) | ({0, floor} if mark_floor else {0}))
    ax.yaxis.set_major_locator(FixedLocator(all_ticks))

    def _format_ytick(y, pos):
        if y == 0:
            return "0"
        if mark_floor and math.isclose(y, floor, rel_tol=1e-9):
            return f"{floor:.3g}"
        exp = math.log10(y)
        if math.isclose(exp, round(exp), abs_tol=1e-6):
            return f"$10^{{{int(round(exp))}}}$"
        return ""

    ax.yaxis.set_major_formatter(FuncFormatter(_format_ytick))

    if mark_floor:
        for tick_val, tick_label in zip(all_ticks, ax.get_yticklabels()):
            if math.isclose(tick_val, floor, rel_tol=1e-9):
                tick_label.set_fontweight("bold")

    return floor


def make_chart(virus, metric, df):
    value_col = metric["value_col"](virus)
    sub = df[["WEEK_ENDING", virus["perc_col"], value_col]].sort_values("WEEK_ENDING").reset_index(drop=True)
    out_path = f"{virus['slug']}_{metric['out_suffix']}.png"

    n_weeks = len(sub)
    n_with_data = sub[value_col].notna().sum()
    print(f"{virus['title']} ({metric['key']}): {n_weeks} weeks total, {n_with_data} with data -> {out_path}")

    pearson_r, pearson_p, spearman_r, spearman_p, n_paired = compute_correlations(sub, virus["perc_col"], value_col)
    print(f"{virus['title']} ({metric['key']}): " f"Spearman rho={spearman_r:.3f} (p={spearman_p:.2e}) (n={n_paired} weeks)")

    nonzero = sub[value_col][sub[value_col] > 0]
    data_floor = nonzero.min() if len(nonzero) else 0

    # Actual zero-value weeks plotted AT the series' nonzero floor instead of
    # true 0 -- otherwise those weeks still visually crash to the bottom of
    # the axis. Correlations above are computed on the unmodified `sub`.
    plot_values = sub[value_col].replace(0, data_floor)
    was_zero = sub[value_col] == 0

    fig, ax_left = plt.subplots(figsize=(15, 7))
    ax_right = ax_left.twinx()

    plot_gap_aware_series(ax_left, sub["WEEK_ENDING"], plot_values, COLOR_WW, label=metric["series_label"],
                           floor_mask=was_zero)
    plot_gap_aware_series(ax_right, sub["WEEK_ENDING"], sub[virus["perc_col"]], COLOR_ED,
                           label="% ED visits (Statewide)")

    _style_axes(ax_left, ax_right, metric["ylabel"])

    _configure_symlog_axis(ax_left, plot_values, top=plot_values.max() * 1.9)
    _configure_symlog_axis(ax_right, sub[virus["perc_col"]], top=sub[virus["perc_col"]].max() * 1.9, mark_floor=False)

    ax_left.xaxis.set_major_locator(mdates.YearLocator())
    ax_left.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    ax_left.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax_left.set_xlabel("Date (Week ending)", fontsize=18, fontweight="bold")
    ax_left.tick_params(axis="both", labelsize=18)
    ax_right.tick_params(axis="both", labelsize=18)
    box_side = virus.get("box_side", "right")
    box_x, box_ha = (0.015, "left") if box_side == "left" else (0.985, "right")
    ax_left.text(
        box_x, 0.97,
        #f"Pearson r = {pearson_r:.2f} ({_format_p(pearson_p)})\n"
        f"Spearman ρ = {spearman_r:.2f} ({_format_p(spearman_p)})\n"
        f"n = {n_paired} weeks",
        transform=ax_left.transAxes, fontsize=18, va="top", ha=box_ha,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9),
        zorder=5)

    lines_left, labels_left = ax_left.get_legend_handles_labels()
    lines_right, labels_right = ax_right.get_legend_handles_labels()
    ax_left.legend(lines_left + lines_right, labels_left + labels_right,
                   loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=2, frameon=False, fontsize=18)
    metric_title = "Wastewater qPCR Signal" if metric["key"] == "copiesperml" else "Wastewater Sequencing Signal"
    ax_left.set_title(
        f"{virus['title']}: {metric_title} vs. ED Visit Rate, Weekly Rolling Average (Concordant Samples)",
        fontsize=18, fontweight="bold", loc="center", pad=20,
    )
    fig.tight_layout(rect=[0, 0.08, 1, 1])

    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.3)
    fig.savefig(out_path.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"Saved {out_path}")


def main():
    xl = pd.ExcelFile(DATA_PATH)
    for virus in VIRUSES:
        df = xl.parse(virus["sheet"])
        for metric in METRICS:
            make_chart(virus, metric, df)


if __name__ == "__main__":
    main()


SARS-CoV-2 (copiesperml): 234 weeks total, 190 with data -> sars_cov2_qpcr_concordant_vs_clinical_rolling_4wk_nodallas.png
SARS-CoV-2 (copiesperml): Spearman rho=0.799 (p=2.02e-43) (n=190 weeks)
Saved sars_cov2_qpcr_concordant_vs_clinical_rolling_4wk_nodallas.png
SARS-CoV-2 (rpkmf): 234 weeks total, 190 with data -> sars_cov2_sequencing_concordant_vs_clinical_rolling_4wk_nodallas.png
SARS-CoV-2 (rpkmf): Spearman rho=0.900 (p=9.11e-70) (n=190 weeks)
Saved sars_cov2_sequencing_concordant_vs_clinical_rolling_4wk_nodallas.png
Influenza (A+B combined) (copiesperml): 234 weeks total, 190 with data -> flu_qpcr_concordant_vs_clinical_rolling_4wk_nodallas.png
Influenza (A+B combined) (copiesperml): Spearman rho=0.566 (p=1.64e-17) (n=190 weeks)
Saved flu_qpcr_concordant_vs_clinical_rolling_4wk_nodallas.png
Influenza (A+B combined) (rpkmf): 234 weeks total, 190 with data -> flu_sequencing_concordant_vs_clinical_rolling_4wk_nodallas.png
Influenza (A+B combined) (rpkmf): Spearman rho=0.855 (p=1.68e